# Lesson Brief — Model Preparation Saving
<!-- LESSON_BRIEF_COURSE11 -->

**What you will do:** Work through this example top to bottom. Read each section header before running the next code cell.

**Why it matters:** Students learn to package models and expose them locally before any cloud complexity.

**How to run:** Use Python 3.10+. Run cells in order. If you change data or a model above, use **Kernel → Restart & Run All** before trusting later cells.

**Stuck?** See `Course 11/START_HERE.md` and `Course 11/DOCS/REQUIREMENTS_COURSE_11.md`.

---


In [ ]:
import numpy as np
import pickle
import os

print("✅ Libraries imported!")
print("\nPreparing AI Model for Deployment: Training and Saving")
print("=" * 60)

print("\nModel Saving Formats:")
print("  - TensorFlow: SavedModel, HDF5")
print("  - PyTorch: .pth, .pt")
print("  - Scikit-learn: Pickle (.pkl)")
print("  - ONNX: Cross-platform format")
print("  - PMML: XML-based format")

print("\n✅ Model preparation concepts understood!")


## 🌍 Real-World Worked Example — MLflow Experiment Tracking (Production Pattern)

**Industry context:**
- Netflix uses MLflow to track 1000s of A/B test model variants
- Airbnb logs every model training run with parameters, metrics, and artifacts
- Booking.com uses experiment tracking to compare models before deploying to 150M users

We demonstrate **MLflow-style experiment tracking** using Python — the same pattern used in production ML pipelines.


In [ ]:
import json, time, pathlib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

RUNS_LOG = pathlib.Path("/tmp/experiment_runs.jsonl")
runs: list[dict] = []

def log_run(name, params, metrics, tags=None):
    entry = {
        "run_id": f"run_{len(runs):03d}",
        "name": name,
        "params": params,
        "metrics": metrics,
        "tags": tags or {},
        "timestamp": time.time(),
    }
    runs.append(entry)
    with open(RUNS_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")
    return entry

iris = load_iris()
X, y = iris.data, iris.target

models = {
    "LogisticRegression": (LogisticRegression(max_iter=200), {"C": 1.0}),
    "RandomForest_50": (RandomForestClassifier(n_estimators=50), {"n_estimators": 50}),
    "RandomForest_100": (RandomForestClassifier(n_estimators=100), {"n_estimators": 100}),
    "GradientBoosting": (
        GradientBoostingClassifier(n_estimators=50),
        {"n_estimators": 50, "lr": 0.1},
    ),
}

results = []
print("Running experiments...")
for name, (clf, params) in models.items():
    start = time.perf_counter()
    cv_scores = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
    elapsed = time.perf_counter() - start
    metrics = {
        "cv_mean_accuracy": round(float(cv_scores.mean()), 4),
        "cv_std": round(float(cv_scores.std()), 4),
        "training_time_s": round(elapsed, 3),
    }
    log_run(name, params, metrics, tags={"dataset": "iris", "framework": "sklearn"})
    results.append((name, metrics))
    print(
        f"  [{name:25s}]  acc={metrics['cv_mean_accuracy']:.4f} "
        f"+/- {metrics['cv_std']:.4f}  | {elapsed:.2f}s"
    )

names = [r[0].replace("_", " ") for r in results]
means = [r[1]["cv_mean_accuracy"] for r in results]
stds = [r[1]["cv_std"] for r in results]
times = [r[1]["training_time_s"] for r in results]
best = int(np.argmax(means))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bars = axes[0].bar(names, means, yerr=stds, capsize=5, alpha=0.8)
bars[best].set_color("green")
bars[best].set_label("Best model")
axes[0].set_title("Model Comparison (MLflow-style)")
axes[0].set_ylabel("CV Accuracy")
axes[0].legend()
axes[1].bar(names, times, alpha=0.8)
axes[1].set_title("Training Time")
axes[1].set_ylabel("Seconds")
plt.suptitle("Experiment tracking pattern used in production MLOps")
plt.tight_layout()
plt.savefig("/tmp/mlflow_comparison.png", dpi=72)

print(f"Best model: {results[best][0]} (accuracy={means[best]:.4f})")
print(f"Experiment log: {RUNS_LOG}")


## 📚 References & Further Reading

**Cloud ML Platforms:**
- [AWS SageMaker](https://docs.aws.amazon.com/sagemaker/)
- [Google Cloud Vertex AI](https://cloud.google.com/vertex-ai/docs)
- [Azure Machine Learning](https://learn.microsoft.com/en-us/azure/machine-learning/)

**MLOps:**
- [MLflow](https://mlflow.org/) — Open-source experiment tracking
- [Weights & Biases](https://wandb.ai/) — Production MLOps platform

**State-of-the-Art:** Netflix, Airbnb, Uber run 1000+ ML models in production using SageMaker/Vertex AI with full MLflow experiment tracking.


## 📝 Summary

You learned **model packaging and serialization** — converting trained models into deployable artifacts. Pickle is simple but Python-only; ONNX is cross-platform and hardware-optimized. Production systems at Uber, Lyft, and Airbnb use ONNX and TorchScript for portability.


## Did you understand? (about 2 minutes)

<!-- STUDENT_SELF_CHECK_COURSE11 -->

Answer **without scrolling** first, then compare with the notebook.

1. **One sentence:** What is the main deployment idea this notebook taught?
2. **Trace one step:** Name one artifact (file, API route, container, or metric) and what role it plays in production.
3. **One question:** What would you ask if you had to deploy this for real users tomorrow?

If any answer is blank, re-run the notebook slowly (one cell → read output → next cell).
